# MTU Notebook Workflow

Run Mapbox Tileset Uploader from a notebook with one editable configuration cell and reusable helper logic.

Default behavior is **dry run** so you can validate conversion and geometry checks without uploading.

## Fast Start

1. If needed, run Cell 2 to install `mtu` in the current kernel (`INSTALL_MTU = True`).
2. Run Cell 3 to confirm `mtu` is available.
3. Edit only `SETTINGS` in Cell 4.
4. Run all remaining cells top to bottom.
5. For a real upload, set `SETTINGS['dry_run'] = False` in Cell 4 and run Cell 8 again.

You do not need to edit any cells outside Cell 4 (except optional `INSTALL_MTU` in Cell 2).

## Cell Guide

- Cell 1: Overview, instructions, and reference notes (this section).
- Cell 2: Optional package install helper for the active kernel.
- Cell 3: Dependency check for `mtu` in the active kernel.
- Cell 4: User configuration (`SETTINGS`) for auth, source, tileset, zoom, and limits.
- Cell 5: Quick pointer to top-level documentation.
- Cell 6: Imports required by helper and execution cells.
- Cell 7: Reusable helper functions (preflight checks, preview, builder helpers).
- Cell 8: One-click run cell (uses `SETTINGS`, performs upload/dry-run, prints results).
- Cell 9: Quick pointer to top-level documentation.

## Source And Format Notes

- Local file mode: `SETTINGS['source_mode'] = 'file'` and set `SETTINGS['source_file']`.
- URL mode: `SETTINGS['source_mode'] = 'url'` and set `SETTINGS['source_url']`.
- Keep `SETTINGS['format_hint'] = None` to auto-detect format unless you need to force one.

Supported input formats: GeoJSON, TopoJSON, Shapefile ZIP/SHP, GeoPackage, KML/KMZ, FlatGeobuf, GeoParquet, GPX.

## Limits And Cautions

- Required token scopes: `tilesets:read`, `tilesets:write`, `tilesets:list`.
- URL-restricted tokens may fail with `403 Forbidden` in CLI/notebook flows.
- Typical Mapbox zoom range is `0-22`; notebook defaults are `4-8`.
- Workflow default upload cap is 1 GB (`SETTINGS['app_default_upload_cap_gb']`).
- You can opt into Mapbox 20 GB per-file cap with `SETTINGS['use_mapbox_full_upload_cap'] = True`.
- Remaining Mapbox CU cannot be queried from this notebook.
- Higher zoom ranges and heavier datasets may increase billable usage.

## How To Read Output (Cell 8)

- `success`: whether the workflow completed without error.
- `dry_run`: whether the run was simulation or real upload.
- `tileset_id`: target tileset identifier used for the run.
- `steps`: ordered list of pipeline steps executed.
- `warnings`: non-fatal issues to review before production runs.

## Colab: Load Variables And Source Files

### Credentials In Colab (recommended: Secrets)

1. Click the key icon in the left sidebar (`Secrets`).
2. Add `MAPBOX_ACCESS_TOKEN` and `MAPBOX_USERNAME`.
3. Keep them enabled for this notebook.
4. Cell 4 auto-detects Colab and reads these values.

### Alternative: environment variables in a cell

```python
import os
os.environ['MAPBOX_ACCESS_TOKEN'] = 'your-token'
os.environ['MAPBOX_USERNAME'] = 'your-username'
```

### Load source files in Colab

- Upload from local machine:

```python
from google.colab import files
uploaded = files.upload()
```

Then set `SETTINGS['source_mode'] = 'file'` and `SETTINGS['source_file'] = Path('/content/<your-file-name>')` in Cell 4.

- Or use URL source:
  Set `SETTINGS['source_mode'] = 'url'` and `SETTINGS['source_url'] = 'https://...'` in Cell 4.

In [ ]:
# Optional install helper for this notebook kernel.
# Set INSTALL_MTU = True only if Cell 3 says mtu is missing.
import subprocess
import sys

INSTALL_MTU = False
INSTALL_TARGET = "mtu"  # or a pinned version, e.g. "mtu==0.1.0"

if INSTALL_MTU:
    print(f"Installing {INSTALL_TARGET} into: {sys.executable}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", INSTALL_TARGET])
    print("Install complete. Re-run Cell 3 to verify.")
else:
    print("Install skipped. Set INSTALL_MTU = True to install mtu in this kernel.")

# Notes:
# - For local source instead of PyPI package, run in terminal:
#   python -m pip install -e .


In [ ]:
# Dependency check (especially useful in Colab)
import importlib.util

has_mtu = importlib.util.find_spec('mtu') is not None

if has_mtu:
    print('mtu package is available in this kernel.')
else:
    print('mtu package is NOT installed in this kernel.')
    print('Install one of the following, then re-run this notebook:')
    print('  Colab/PyPI:   !pip install mtu')
    print('  Local source: !pip install -e .')
    print('After install, restart the runtime/kernel if import errors persist.')

In [ ]:
# Configuration only: edit values in this cell, then run remaining cells unchanged.
import os
from pathlib import Path

# Credentials (auto-adapt for Colab Secrets, then fallback to environment variables).
try:
    from google.colab import userdata  # type: ignore

    _token_from_colab = userdata.get('MAPBOX_ACCESS_TOKEN') or ''
    _username_from_colab = userdata.get('MAPBOX_USERNAME') or ''
except Exception:
    _token_from_colab = ''
    _username_from_colab = ''

SETTINGS = {
    # Auth
    'mapbox_access_token': _token_from_colab or os.environ.get('MAPBOX_ACCESS_TOKEN', ''),
    'mapbox_username': _username_from_colab or os.environ.get('MAPBOX_USERNAME', ''),

    # Run behavior
    'dry_run': True,

    # Source settings
    'source_mode': os.environ.get('MTU_SOURCE_MODE', 'file').strip().lower(),  # 'file' or 'url'
    'source_file': Path('.../data.geojson'),
    'source_url': 'https://example.com/data.geojson',
    'format_hint': None,  # None enables auto-detect

    # Tileset settings
    'tileset_id': 'demo-tileset-id',
    'tileset_name': 'Demo Tileset',
    'layer_name': 'data',
    'min_zoom': 4,
    'max_zoom': 8,

    # Upload limits
    'mapbox_zoom_min': 0,
    'mapbox_zoom_max': 22,
    'app_default_upload_cap_gb': 1,
    'mapbox_full_upload_cap_gb': 20,
    'use_mapbox_full_upload_cap': False,

    # Optional capacity planning (set to 0 to ignore)
    'capacity_limit_mb': 0.0,
    'capacity_used_mb': 0.0,
}

if SETTINGS['source_mode'] not in {'file', 'url'}:
    raise ValueError("SETTINGS['source_mode'] must be 'file' or 'url'")

print('Configuration loaded. Update SETTINGS above only.')
print(f"dry_run={SETTINGS['dry_run']}, source_mode={SETTINGS['source_mode']}")
print(
    f"zoom={SETTINGS['min_zoom']}-{SETTINGS['max_zoom']} (Mapbox typical "
    f"{SETTINGS['mapbox_zoom_min']}-{SETTINGS['mapbox_zoom_max']})"
)
print(
    'upload_cap_mode=' + (
        'Mapbox full 20 GB' if SETTINGS['use_mapbox_full_upload_cap'] else 'App default 1 GB'
    )
)
print(f"MAPBOX_ACCESS_TOKEN set: {bool(SETTINGS['mapbox_access_token'])}")
print(f"MAPBOX_USERNAME set: {bool(SETTINGS['mapbox_username'])}")

## Reference

> Key documentation, limits, and supported formats are now at the top in Cell 1.

If you need to change behavior, update `SETTINGS` in Cell 4 only.

In [ ]:
import json
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from mtu.uploader import TilesetConfig, TilesetUploader

In [ ]:
# Reusable helpers. No user edits needed in this cell.
def build_tileset_config(settings: dict[str, Any]) -> TilesetConfig:
    return TilesetConfig(
        tileset_id=str(settings['tileset_id']),
        tileset_name=str(settings['tileset_name']),
        layer_name=str(settings['layer_name']),
        min_zoom=int(settings['min_zoom']),
        max_zoom=int(settings['max_zoom']),
    )


def build_uploader(settings: dict[str, Any]) -> TilesetUploader:
    return TilesetUploader(
        validate_geometry=True,
        use_mapbox_full_upload_cap=bool(settings['use_mapbox_full_upload_cap']),
    )


def get_active_upload_cap_gb(settings: dict[str, Any]) -> float:
    return float(
        settings['mapbox_full_upload_cap_gb']
        if settings['use_mapbox_full_upload_cap']
        else settings['app_default_upload_cap_gb']
    )


def describe_source(settings: dict[str, Any]) -> None:
    cap_gb = get_active_upload_cap_gb(settings)
    cap_mb = cap_gb * 1024
    print(f'Active upload cap: {cap_gb} GB')

    source_mode = str(settings['source_mode'])
    if source_mode == 'url':
        source_url = str(settings['source_url'])
        print(f'URL source selected: {source_url}')
        parsed = urlparse(source_url)
        if not parsed.scheme or not parsed.netloc:
            raise ValueError('SOURCE_URL must be a valid absolute URL when source_mode is url')
        if float(settings['capacity_limit_mb']) > 0:
            print('Note: local size/capacity projection is skipped for URL sources.')
        return

    source_file = Path(settings['source_file'])
    if not source_file.exists():
        print(f'WARNING: file not found: {source_file}')
        return

    size_mb = source_file.stat().st_size / (1024 * 1024)
    print(f'Input size: {size_mb:.2f} MB')
    if size_mb > cap_mb:
        print('WARNING: input file exceeds active upload cap.')

    capacity_limit_mb = float(settings['capacity_limit_mb'])
    if capacity_limit_mb > 0:
        projected = float(settings['capacity_used_mb']) + size_mb
        print(f'Projected usage: {projected:.2f} / {capacity_limit_mb:.2f} MB')
        if projected > capacity_limit_mb:
            print('WARNING: projected usage exceeds configured capacity limit.')

    if source_file.suffix.lower() not in {'.geojson', '.json'}:
        return

    # Optional map preview for local GeoJSON files using folium.
    try:
        import folium
        from IPython.display import display

        with source_file.open('r', encoding='utf-8') as f:
            geojson_data: Any = json.load(f)

        m = folium.Map(location=[0, 0], zoom_start=int(settings['min_zoom']), control_scale=True)
        layer = folium.GeoJson(geojson_data, name='source data')
        layer.add_to(m)

        try:
            bounds = layer.get_bounds()
            if bounds and len(bounds) == 2:
                sw, ne = bounds
                if all(v is not None for v in [sw[0], sw[1], ne[0], ne[1]]):
                    m.fit_bounds([[float(sw[0]), float(sw[1])], [float(ne[0]), float(ne[1])]])
        except Exception:
            pass

        display(m)
    except Exception as ex:
        print('Map preview skipped (folium not installed or invalid GeoJSON):', ex)

Active upload cap: 1 GB
Input size: 7.38 MB


In [ ]:
# One-click execution cell: run after updating SETTINGS in Cell 4.
describe_source(SETTINGS)

config = build_tileset_config(SETTINGS)
uploader = build_uploader(SETTINGS)

if SETTINGS['source_mode'] == 'url':
    result = uploader.upload_from_url(
        url=str(SETTINGS['source_url']),
        config=config,
        format_hint=SETTINGS['format_hint'],
        dry_run=bool(SETTINGS['dry_run']),
    )
else:
    result = uploader.upload_from_file(
        file_path=Path(SETTINGS['source_file']),
        config=config,
        format_hint=SETTINGS['format_hint'],
        dry_run=bool(SETTINGS['dry_run']),
    )

print('success:', result.success)
print('dry_run:', result.dry_run)
print('tileset_id:', result.tileset_id)
print('steps:', result.steps)
print('warnings:', result.warnings)

success: True
dry_run: True
tileset_id: ocha-rosea-1.demo-tileset-id
steps: {'convert': True, 'validate': True}
warnings: []


## Reference

> Real upload and output interpretation notes are documented at the top in Cell 1.

For production runs: set `SETTINGS['dry_run'] = False` in Cell 4, then rerun Cell 8.